# DTHU Search — triển khai trên Google Colab
Giao diện giữ nguyên theo `index.html` bạn gửi. Chạy các cell theo thứ tự. **Bật GPU T4** nếu có. Runtime Colab ngắt thì server cũng ngắt. Micro có thể bị giới hạn bởi quyền trình duyệt/Colab Proxy.

In [1]:
!pip -q install fastapi uvicorn python-multipart transformers accelerate bm25s underthesea pandas
!apt-get -qq update && apt-get -qq install -y ffmpeg

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


In [2]:
from google.colab import files
from pathlib import Path
Path('/content/static').mkdir(exist_ok=True)
print('Upload index.html và dthu_all.csv (hai file trong gói ZIP)')

Upload index.html và dthu_all.csv (hai file trong gói ZIP)


In [4]:
!mv index.html static
!mv dthu-logo.png static

In [5]:
import os, re, tempfile, threading
from pathlib import Path
from contextlib import asynccontextmanager
import bm25s, pandas as pd, torch
from transformers import pipeline
from underthesea import word_tokenize
from fastapi import FastAPI, File, UploadFile, Query, HTTPException
from fastapi.responses import FileResponse
from fastapi.staticfiles import StaticFiles
from pydantic import BaseModel, Field
from starlette.concurrency import run_in_threadpool

BASE=Path('/content')
CSV_PATH=BASE/'dthu_all.csv'
STATIC=BASE/'static'
MODEL_NAME=os.getenv('ASR_MODEL','vinai/PhoWhisper-small')
MIN_ID=int(os.getenv('MIN_ID','1000'))
MAX_AUDIO_BYTES=20*1024*1024
EXTENSIONS={'.wav','.mp3','.m4a','.webm','.ogg','.flac','.mp4'}
DIGITS={'không':0,'một':1,'mốt':1,'hai':2,'ba':3,'bốn':4,'tư':4,'năm':5,'lăm':5,'sáu':6,'bảy':7,'tám':8,'chín':9}
UNITS='|'.join(sorted(DIGITS,key=len,reverse=True))
YEAR_WORDS=re.compile(rf'\bnăm\s+((?:hai\s+nghìn(?:\s+không\s+trăm)?(?:\s+hai\s+mươi)?(?:\s+(?:{UNITS}))?|hai\s+không\s+hai\s+(?:{UNITS})|20\d{{2}}|\d{{2}}))\b',re.I)

def parse_year(value):
    value=re.sub(r'\s+',' ',value.lower()).strip()
    if re.fullmatch(r'20\d{2}',value): return value
    if re.fullmatch(r'\d{2}',value): return '20'+value
    parts=value.split()
    if len(parts)==4 and all(x in DIGITS for x in parts):
        year=''.join(str(DIGITS[x]) for x in parts)
        return year if year.startswith('20') else None
    if value.startswith('hai nghìn'):
        rest=value[len('hai nghìn'):].strip()
        if rest.startswith('không trăm'): rest=rest[len('không trăm'):].strip()
        tens=0
        if rest.startswith('hai mươi'):
            tens=20
            rest=rest[len('hai mươi'):].strip()
        if not rest or rest in DIGITS:
            return str(2000+tens+(DIGITS[rest] if rest else 0))
    return None

def normalize_year(query):
    return YEAR_WORDS.sub(lambda m:'năm '+(parse_year(m.group(1)) or m.group(1)),query.lower())

def tokenizer(text): return word_tokenize(str(text).lower(),format='text').split()

class TextRequest(BaseModel):
    query:str=Field(min_length=1)
    top_k:int=Field(default=5,ge=1,le=100)
class SearchResult(BaseModel):
    id:int
    title:str
    url:str
    score:float
class SearchResponse(BaseModel):
    query:str
    normalized_query:str
    tokens:list[str]
    total:int
    results:list[SearchResult]
    transcription:str|None=None

@asynccontextmanager
async def lifespan(app):
    df=pd.read_csv(CSV_PATH,encoding='utf-8-sig')
    for col in ('id','title','url'):
        if col not in df.columns: raise RuntimeError('Missing CSV column: '+col)
    df['id']=pd.to_numeric(df['id'],errors='coerce')
    df=df.dropna(subset=['id','title','url'])
    df=df[df.id>MIN_ID].reset_index(drop=True)
    if df.empty: raise RuntimeError('No searchable rows')
    retriever=bm25s.BM25()
    retriever.index([tokenizer(title) for title in df.title])
    device=0 if torch.cuda.is_available() else -1
    asr=pipeline('automatic-speech-recognition',model=MODEL_NAME,device=device)
    app.state.df=df;app.state.retriever=retriever;app.state.asr=asr
    app.state.asr_lock=threading.Lock()
    print('Ready:',len(df),'documents; ASR:',MODEL_NAME,'GPU:',device==0)
    yield
app=FastAPI(title='DThU Search API',lifespan=lifespan)
app.mount('/static',StaticFiles(directory=str(STATIC)),name='static')
@app.get('/',include_in_schema=False)
def home(): return FileResponse(STATIC/'index.html')
@app.get('/api/health')
def health():
    return {'status':'ok','documents':len(app.state.df),'asr_model':MODEL_NAME}

def search(query, top_k, transcription=None):
    normalized = normalize_year(query.strip())
    tokens = tokenizer(normalized)
    items = []

    BM25_THRESHOLD = 2.0

    if tokens:
        indexes, scores = app.state.retriever.retrieve(
            [tokens],
            k=min(top_k, len(app.state.df))
        )

        for idx, score in zip(indexes[0], scores[0]):

            # Chỉ lấy kết quả BM25 > 2.0
            if float(score) <= BM25_THRESHOLD:
                continue

            row = app.state.df.iloc[int(idx)]

            items.append(
                SearchResult(
                    id=int(row['id']),
                    title=str(row['title']),
                    url=str(row['url']),
                    score=float(score)
                )
            )

        # Sắp xếp theo ID giảm dần
        items.sort(key=lambda x: x.id, reverse=True)

    return SearchResponse(
        query=query,
        normalized_query=normalized,
        tokens=tokens,
        total=len(items),
        results=items,
        transcription=transcription
    )

@app.post('/api/search/text',response_model=SearchResponse)
def text_search(payload:TextRequest): return search(payload.query,payload.top_k)
@app.post('/api/search/audio',response_model=SearchResponse)
async def audio_search(file:UploadFile=File(...),top_k:int=Query(5,ge=1,le=100)):
    ext=Path(file.filename or '').suffix.lower()
    if ext not in EXTENSIONS: raise HTTPException(415,'Unsupported audio extension')
    path=None
    try:
        with tempfile.NamedTemporaryFile(suffix=ext,delete=False) as tmp:
            path=tmp.name;total=0
            while chunk:=await file.read(1024*1024):
                total+=len(chunk)
                if total>MAX_AUDIO_BYTES: raise HTTPException(413,'Audio exceeds 20 MB')
                tmp.write(chunk)
        if total==0: raise HTTPException(400,'Empty audio file')
        def transcribe():
            with app.state.asr_lock: return app.state.asr(path)['text'].strip()
        try: transcription=await run_in_threadpool(transcribe)
        except Exception as exc: raise HTTPException(422,'Cannot decode/transcribe audio') from exc
        return search(transcription,top_k,transcription)
    finally:
        await file.close()
        if path and os.path.exists(path): os.unlink(path)


In [6]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

In [12]:
import threading
import time
import requests
import uvicorn

PORT = 8000

def run_server():
    uvicorn.run(
        app,
        host="0.0.0.0",
        port=PORT,
        log_level="info"
    )

server_thread = threading.Thread(
    target=run_server,
    daemon=True
)

server_thread.start()

print("Đang khởi động FastAPI...")

for i in range(180):
    try:
        response = requests.get(
            f"http://127.0.0.1:{PORT}/api/health",
            timeout=3
        )

        if response.ok:
            print("✅ FastAPI đã sẵn sàng!")
            print(response.json())
            break

    except requests.exceptions.RequestException:
        pass

    time.sleep(2)

else:
    print("❌ Server chưa khởi động thành công.")

Đang khởi động FastAPI...


INFO:     Started server process [1546]
INFO:     Waiting for application startup.
DEBUG:bm25s:Building index from tokens


BM25S Create Vocab:   0%|          | 0/473 [00:00<?, ?it/s]

BM25S Convert tokens to indices:   0%|          | 0/473 [00:00<?, ?it/s]

BM25S Count Tokens:   0%|          | 0/473 [00:00<?, ?it/s]

...

added_tokens.json:   0%|          | 0.00/2.08k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.08k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/339 [00:00<?, ?B/s]

INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


Ready: 473 documents; ASR: vinai/PhoWhisper-small GPU: True
INFO:     127.0.0.1:38532 - "GET /api/health HTTP/1.1" 200 OK
✅ FastAPI đã sẵn sàng!
{'status': 'ok', 'documents': 473, 'asr_model': 'vinai/PhoWhisper-small'}


In [13]:
import subprocess
import time
import re

tunnel = subprocess.Popen(
    [
        "cloudflared",
        "tunnel",
        "--url",
        "http://127.0.0.1:8000",
        "--no-autoupdate"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

public_url = None

for _ in range(60):
    line = tunnel.stdout.readline()
    print(line.strip())

    match = re.search(
        r"https://[a-zA-Z0-9-]+\.trycloudflare\.com",
        line
    )

    if match:
        public_url = match.group(0)
        break

print("DTHU Search URL:", public_url)

2026-10-30T05:06:03Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-10-30T05:06:03Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-10-30T05:06:07Z INF +--------------------------------------------------------------------------------------------+
2026-10-30T05:06:07Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-10-30T05:06:07Z INF |  https://sees-cartoon-intended-chemicals.trycloudflare